In [ ]:
import pandas as pd
import numpy as np
from efficient_apriori import apriori

In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
#Update this one last time before performing analysis. Might have changed.
datas = pd.read_csv('PSCompPars_2025.06.12_09.54.16.csv', comment='#', low_memory=False)
data = pd.DataFrame(data=datas)
#These two were labeled incorrectly in the dataframe
data.replace({ 'st_spectype': {'m3 V':'M V'} }, inplace = True)
data.replace({ 'st_spectype': {'sdBV':'B V'} }, inplace = True)
#print(data.count())
#4673 = m -> M V
#5209 = s -> B V

In [41]:
#estimates the semimajor axis given a row of a data frame given stellar mass and orbital period: Kepler's 3rd law
#Only estimates the semimajor axis if the value is originally NaN
def estimate_orbsmax(row):
    if pd.isna(row['pl_orbsmax']):
        try: 
            return (((6.67e-11 * (row['st_mass'] * 1.989e30) * ((row['pl_orbper'] * 86400)**2)) / (4 * (np.pi**2)))**(1/3)) / 1.496e11
        except:
            return np.nan
    else:
        return row['pl_orbsmax']

#Sets the spectral type if it is already in the string for the spectype variable for each row.
#Otherwise, it is estimated based on equillibrium temperature.
def get_spectype(row):
    try: 
        return row['st_spectype'][0]
    except:
        if(row['st_teff']>=30000):
            return 'O'
        elif(row['st_teff']>=10000):
            return 'B'
        elif(row['st_teff']>=7500):
            return 'A'
        elif(row['st_teff']>=6000):
            return 'F'
        elif(row['st_teff']>=5200):
            return 'G'
        elif(row['st_teff']>=3700):
            return 'K'
        elif(row['st_teff']>=2400):
            return 'M'
        else:
            return row['st_spectype']

#Estimates the equillibrium temperature of a row given stellar luminosity, and semimajor axis
#Only estimates if there is a missing value for pl_eqt: Stefan-Boltzmann's law
def estimate_pl_eqt(row):
    if pd.isna(row['pl_eqt']):
        try:
            return (((10 ** row['st_lum']) * 3.827e26) / (16 * np.pi * 5.67e-8 * ((row['pl_orbsmax'] * 1.496e11) ** 2))) ** (1 / 4)
        except:
            return np.nan
    else:
        return row['pl_eqt']
#Grabs the last value of the string, usually the Luminosity Class of the star
#Otherwise, estimates whethoer or not the star is main sequence based on the luminosity of the star
def is_mainsequence(row):
    if(str(row['st_spectype'])[-1] == ('V') and str(row['st_spectype'])[-2:] != ('IV')):
        return 'True'
    else:
        try:
            if(str(row['spectype']).__contains__('M') and row['st_lum'] < -1.0969):
                return 'True'
            elif(str(row['spectype']).__contains__('K') and row['st_lum'] < -0.2218):
                return 'True'
            elif(str(row['spectype']).__contains__('G') and row['st_lum'] < 0.1761):
                return 'True'
            elif(str(row['spectype']).__contains__('F') and row['st_lum'] < 0.699):
                return 'True'
            elif(str(row['spectype']).__contains__('A') and row['st_lum'] < 1.3979):
                return 'True'
            elif(str(row['spectype']).__contains__('B') and row['st_lum'] < 4.477):
                return 'True'
            elif(str(row['spectype']).__contains__('O')):
                return 'True'
            else:
                return 'False'
        except:
            return 'False'
                
    
#Applying the above functions to the dataframe.
data['pl_orbsmax'] = data.apply(estimate_orbsmax, axis=1)
data['pl_eqt'] = data.apply(estimate_pl_eqt, axis = 1)
data['spectype'] = data.apply(get_spectype, axis = 1)
data['isMS'] = data.apply(is_mainsequence, axis = 1)

In [ ]:
#Remove systems with more that one host star
star = data.loc[data['sy_snum'] == 1]
# Excluded planet systems that did not have any data on the variables of interest
star = star.loc[star['pl_orbper'].isna() == False]
star = star.loc[star['pl_rade'].isna() == False]
star = star.loc[star['st_teff'].isna() == False]
star = star.loc[star['pl_eqt'].isna() == False]
star = star.loc[star['spectype'].isna() == False]

In [ ]:
#Puts planets into there designated categories. Also filters the ones that are main sequence into a seporate dataframe.
star['pl_rad_bin'] = pd.cut(star['pl_rade'], bins = [0, 1.7, 3.9, 9.4, 1000], labels = ['Terrestrial', 'Mini-Neptune', 'Sub-Saturn', 'Gas Giant'])
star['pl_orb_bin'] = pd.cut(star['pl_orbper'], bins = [0, 10, 100, 1000, 10000000000], labels = ['Short Orbital', 'Medium-Short Orbital', 'Medium-Long Orbital', 'Long Orbital'])
star['pl_eqt_bin'] = pd.cut(star['pl_eqt'], bins = [0, 200, 400, 1000, 10000000], labels = ['Cold', 'Temperate', 'Warm', 'Hot'])
star['pl_class'] = star[['pl_eqt_bin','pl_rad_bin', 'pl_orb_bin']].astype('str').agg(" ".join, axis=1)
star['stars_class'] = star[['spectype', 'pl_class']].agg(" ".join, axis=1)
onlyMS = star.loc[(star['isMS'] != 'False')]

In [45]:
#Creating a dataframe with only main sequence stars and their proper classificaiton
testMSpl = {'type': onlyMS['pl_class'],'host':onlyMS['hostname']}
testingMSpl = pd.DataFrame(data=testMSpl)
#Put into tuples by host star for the package
baseMSpl = testingMSpl.groupby('host')    
MSplAA = [tuple(baseMSpl.get_group(x)['type']) for x in baseMSpl.groups]
#Performs the association analysis based on the planet classifications.
itemsetMSpl, rulesMSpl = apriori(MSplAA, min_support= 2/len(onlyMS), min_confidence=.5)
print('There are ' + str(rulesMSpl.__len__())+ ' rules.')
print(rulesMSpl)

There are 28 rules.
[{Cold Terrestrial Medium-Short Orbital} -> {Temperate Terrestrial Medium-Short Orbital}, {Cold Terrestrial Medium-Short Orbital} -> {Temperate Terrestrial Short Orbital}, {Warm Sub-Saturn Medium-Long Orbital} -> {Warm Mini-Neptune Medium-Short Orbital}, {Cold Gas Giant Medium-Long Orbital, Warm Mini-Neptune Short Orbital} -> {Cold Gas Giant Long Orbital}, {Cold Gas Giant Long Orbital, Warm Mini-Neptune Short Orbital} -> {Cold Gas Giant Medium-Long Orbital}, {Cold Gas Giant Long Orbital, Temperate Mini-Neptune Medium-Long Orbital} -> {Warm Mini-Neptune Medium-Short Orbital}, {Cold Gas Giant Long Orbital, Warm Mini-Neptune Short Orbital} -> {Warm Mini-Neptune Medium-Short Orbital}, {Hot Mini-Neptune Medium-Short Orbital, Warm Sub-Saturn Medium-Short Orbital} -> {Warm Mini-Neptune Medium-Short Orbital}, {Hot Mini-Neptune Short Orbital, Temperate Sub-Saturn Medium-Long Orbital} -> {Warm Sub-Saturn Medium-Short Orbital}, {Warm Sub-Saturn Medium-Short Orbital, Warm Sub-S

In [48]:
#creates a new dataset with only the classification
testplanets = {'type': star['pl_class'],'host':star['hostname']}
testingPlanets = pd.DataFrame(data=testplanets)
#Put into tuples by host star for the package
base = testingPlanets.groupby('host')    
planetAA = [tuple(base.get_group(x)['type']) for x in base.groups]
#Performs the association analysis based on the planet classifications.
itemsetspl, rulespl = apriori(planetAA, min_support= 2/len(star), min_confidence=.5)
print('There are ' + str(rulespl.__len__())+ ' rules.')
print(rulespl)

There are 29 rules.
[{Cold Terrestrial Medium-Short Orbital} -> {Temperate Terrestrial Medium-Short Orbital}, {Cold Terrestrial Medium-Short Orbital} -> {Temperate Terrestrial Short Orbital}, {Warm Sub-Saturn Medium-Long Orbital} -> {Warm Mini-Neptune Medium-Short Orbital}, {Cold Gas Giant Medium-Long Orbital, Warm Mini-Neptune Short Orbital} -> {Cold Gas Giant Long Orbital}, {Cold Gas Giant Long Orbital, Warm Mini-Neptune Short Orbital} -> {Cold Gas Giant Medium-Long Orbital}, {Cold Gas Giant Long Orbital, Temperate Mini-Neptune Medium-Long Orbital} -> {Warm Mini-Neptune Medium-Short Orbital}, {Cold Gas Giant Long Orbital, Warm Mini-Neptune Short Orbital} -> {Warm Mini-Neptune Medium-Short Orbital}, {Hot Mini-Neptune Medium-Short Orbital, Hot Terrestrial Short Orbital} -> {Warm Mini-Neptune Medium-Short Orbital}, {Hot Mini-Neptune Medium-Short Orbital, Warm Sub-Saturn Medium-Short Orbital} -> {Warm Mini-Neptune Medium-Short Orbital}, {Hot Mini-Neptune Short Orbital, Temperate Sub-Satu

In [ ]:
# adding the gas stars only adds one extra rule (29 veresus 28). I was curious about which rule this was.
for rule in rulespl:
        if(rulesMSpl.__contains__(rule)):
                print("same")
        else:
                print(rule)


same
same
same
same
same
same
same
{Hot Mini-Neptune Medium-Short Orbital, Hot Terrestrial Short Orbital} -> {Warm Mini-Neptune Medium-Short Orbital} (conf: 0.750, supp: 0.001, lift: 2.984, conv: 2.995)
same
same
same
same
same
same
same
same
same
same
same
same
same
same
same
same
same
same
same
same
same


In [53]:
testMSplst = {'type': onlyMS['stars_class'],'host':onlyMS['hostname']}
testingMSplst = pd.DataFrame(data=testMSplst)
#Put into tuples by host star for the package
baseMSplst = testingMSplst.groupby('host')    
MSplstAA = [tuple(baseMSplst.get_group(x)['type']) for x in baseMSplst.groups]
#Performs the association analysis based on the planet classifications.
itemsetMSplst, rulesMSplst = apriori(MSplstAA, min_support= 2/len(onlyMS), min_confidence=.5)
print('There are ' + str(rulesMSplst.__len__())+ ' rules.')
rulesMSplst

There are 56 rules.


[{F Warm Terrestrial Short Orbital} -> {F Hot Terrestrial Short Orbital},
 {F Warm Sub-Saturn Medium-Long Orbital} -> {F Warm Mini-Neptune Medium-Short Orbital},
 {F Warm Terrestrial Short Orbital} -> {F Warm Mini-Neptune Medium-Short Orbital},
 {G Cold Gas Giant Medium-Long Orbital} -> {G Hot Gas Giant Short Orbital},
 {G Cold Gas Giant Medium-Long Orbital} -> {G Temperate Gas Giant Medium-Long Orbital},
 {G Cold Sub-Saturn Long Orbital} -> {G Hot Mini-Neptune Short Orbital},
 {G Warm Mini-Neptune Medium-Long Orbital} -> {G Hot Mini-Neptune Short Orbital},
 {G Warm Sub-Saturn Medium-Long Orbital} -> {G Temperate Sub-Saturn Medium-Long Orbital},
 {M Cold Sub-Saturn Medium-Long Orbital} -> {M Temperate Mini-Neptune Medium-Short Orbital},
 {M Temperate Sub-Saturn Medium-Short Orbital} -> {M Cold Sub-Saturn Medium-Long Orbital},
 {M Cold Sub-Saturn Medium-Long Orbital} -> {M Temperate Sub-Saturn Medium-Short Orbital},
 {M Cold Sub-Saturn Medium-Long Orbital} -> {M Warm Mini-Neptune Short 

In [55]:
#creates a new dataset with only the classification
testplst = {'type': star['stars_class'],'host':star['hostname']}
testingplst = pd.DataFrame(data=testplst)
#Put into tuples by host star for the package
basis = testingplst.groupby('host')    
PlStAA = [tuple(basis.get_group(x)['type']) for x in basis.groups]
#Performs the association analysis based on the planet classifications and the stellar classifications.
itemsetsstpl, rulesstpl = apriori(PlStAA, min_support= 2/len(star), min_confidence=.5)
print("There are " + str(rulesstpl.__len__())+ " rules.")
rulesstpl

There are 53 rules.


[{F Warm Terrestrial Short Orbital} -> {F Hot Terrestrial Short Orbital},
 {F Warm Terrestrial Short Orbital} -> {F Warm Mini-Neptune Medium-Short Orbital},
 {G Cold Gas Giant Medium-Long Orbital} -> {G Hot Gas Giant Short Orbital},
 {G Cold Gas Giant Medium-Long Orbital} -> {G Temperate Gas Giant Medium-Long Orbital},
 {G Cold Sub-Saturn Long Orbital} -> {G Hot Mini-Neptune Short Orbital},
 {G Hot Terrestrial Medium-Short Orbital} -> {G Hot Terrestrial Short Orbital},
 {G Warm Sub-Saturn Medium-Long Orbital} -> {G Temperate Sub-Saturn Medium-Long Orbital},
 {G Warm Sub-Saturn Medium-Long Orbital} -> {G Warm Mini-Neptune Medium-Short Orbital},
 {M Cold Terrestrial Medium-Short Orbital} -> {M Temperate Terrestrial Medium-Short Orbital},
 {M Cold Terrestrial Medium-Short Orbital} -> {M Temperate Terrestrial Short Orbital},
 {F Hot Mini-Neptune Medium-Short Orbital, F Warm Sub-Saturn Medium-Short Orbital} -> {F Warm Mini-Neptune Medium-Short Orbital},
 {F Hot Terrestrial Short Orbital, F 

In [ ]:
#comparing the rules for the main sequence stars to the rules with all stars. extras are for all stars.
for rule in rulesstpl:
        if(rulesMSplst.__contains__(rule)):
                print("same")
        else:
                print(rule)

same
same
same
same
same
{G Hot Terrestrial Medium-Short Orbital} -> {G Hot Terrestrial Short Orbital} (conf: 0.667, supp: 0.001, lift: 9.061, conv: 2.779)
same
{G Warm Sub-Saturn Medium-Long Orbital} -> {G Warm Mini-Neptune Medium-Short Orbital} (conf: 0.500, supp: 0.001, lift: 3.379, conv: 1.704)
same
same
same
same
{F Hot Mini-Neptune Short Orbital, G Warm Mini-Neptune Medium-Short Orbital} -> {F Warm Mini-Neptune Medium-Short Orbital} (conf: 1.000, supp: 0.001, lift: 20.804, conv: 951933404.941)
same
same
same
same
same
same
{G Hot Mini-Neptune Medium-Short Orbital, G Warm Mini-Neptune Medium-Short Orbital} -> {G Hot Terrestrial Short Orbital} (conf: 0.500, supp: 0.001, lift: 6.796, conv: 1.853)
{G Hot Mini-Neptune Medium-Short Orbital, G Hot Terrestrial Short Orbital} -> {G Warm Mini-Neptune Medium-Short Orbital} (conf: 0.667, supp: 0.001, lift: 4.506, conv: 2.556)
same
same
same
same
same
same
same
same
same
same
same
same
same
same
same
same
same
same
same
same
same
same
same
sa

In [59]:
for rule in rulesMSplst:
        if(rulesstpl.__contains__(rule)):
                print("same")
        else:
                print(rule)

same
{F Warm Sub-Saturn Medium-Long Orbital} -> {F Warm Mini-Neptune Medium-Short Orbital} (conf: 0.500, supp: 0.001, lift: 8.917, conv: 1.888)
same
same
same
same
{G Warm Mini-Neptune Medium-Long Orbital} -> {G Hot Mini-Neptune Short Orbital} (conf: 0.556, supp: 0.002, lift: 11.721, conv: 2.143)
same
{M Cold Sub-Saturn Medium-Long Orbital} -> {M Temperate Mini-Neptune Medium-Short Orbital} (conf: 0.500, supp: 0.001, lift: 27.741, conv: 1.964)
{M Temperate Sub-Saturn Medium-Short Orbital} -> {M Cold Sub-Saturn Medium-Long Orbital} (conf: 0.500, supp: 0.001, lift: 374.500, conv: 1.997)
{M Cold Sub-Saturn Medium-Long Orbital} -> {M Temperate Sub-Saturn Medium-Short Orbital} (conf: 0.500, supp: 0.001, lift: 374.500, conv: 1.997)
{M Cold Sub-Saturn Medium-Long Orbital} -> {M Warm Mini-Neptune Short Orbital} (conf: 0.500, supp: 0.001, lift: 19.973, conv: 1.950)
same
same
{M Warm Terrestrial Medium-Short Orbital} -> {M Warm Mini-Neptune Short Orbital} (conf: 0.500, supp: 0.001, lift: 19.973,